# MVP Playground

This notebook is a self-contained first-pass prototype for the human / robot / AI music project.

It is intentionally MIDI-only and rule-based:

- load an input MIDI file
- extract a small set of musical features
- choose a response strategy
- generate a reactive MIDI response
- write the result to disk


In [17]:
from pathlib import Path
import numpy as np
import pretty_midi

SEED = 7
np.random.seed(SEED)

def find_repo_root():
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidate in candidates:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'data').exists():
            return candidate
    raise FileNotFoundError('Could not find repo root from current working directory')

ROOT = find_repo_root()
INPUT_MIDI = ROOT / 'data' / 'input_midi' / 'mvp_minimalist_input_variant.mid'
OUTPUT_MIDI = ROOT / 'data' / 'output_midi' / 'mvp_minimalist_response.mid'
TEMPO = 120
TIME_SIG = (4, 4)
LATENCY_BARS = 1

INPUT_MIDI, OUTPUT_MIDI


(PosixPath('/Users/adrientalbot/Desktop/ai-jam-partner/data/input_midi/mvp_minimalist_input_variant.mid'),
 PosixPath('/Users/adrientalbot/Desktop/ai-jam-partner/data/output_midi/mvp_minimalist_response.mid'))

## Load Input

The MVP assumes a single MIDI track for the first pass, but it will fall back to the first track if multiple instruments are present.


In [18]:
midi = pretty_midi.PrettyMIDI(str(INPUT_MIDI))
if not midi.instruments:
    raise ValueError('Input MIDI contains no instruments')

instrument = midi.instruments[0]
notes = sorted(instrument.notes, key=lambda n: n.start)
source_instrument_name = pretty_midi.program_to_instrument_name(instrument.program)

print(f'Instruments: {len(midi.instruments)}')
print(f'Source instrument: {source_instrument_name}')
print(f'Notes: {len(notes)}')
print(f'Pitch range: {min(n.pitch for n in notes)}-{max(n.pitch for n in notes)}')
print(f'Duration: {midi.get_end_time():.2f}s')


Instruments: 1
Source instrument: Acoustic Grand Piano
Notes: 126
Pitch range: 45-81
Duration: 49.34s


## Feature Extraction

The first version only needs a small musical summary: density, register, phrase length, and a short motif from the tail of the phrase.


In [19]:
def seconds_per_bar(tempo=TEMPO, time_sig=TIME_SIG):
    return 60.0 / tempo * time_sig[0]

def density_bucket(notes_per_bar):
    if notes_per_bar < 4:
        return 'low'
    if notes_per_bar < 7:
        return 'medium'
    return 'high'

def register_bucket(avg_pitch):
    if avg_pitch < 54:
        return 'low'
    if avg_pitch < 66:
        return 'mid'
    return 'high'

def extract_features(notes, tempo=TEMPO, time_sig=TIME_SIG):
    pitches = [n.pitch for n in notes]
    avg_pitch = float(np.mean(pitches))
    pitch_span = max(pitches) - min(pitches) if pitches else 0
    duration = notes[-1].end - notes[0].start if notes else 0.0
    bars = max(1, int(round(duration / seconds_per_bar(tempo, time_sig))))
    notes_per_bar = len(notes) / bars if bars else len(notes)
    tail = sorted(notes, key=lambda n: n.start)[-8:]
    motif = [n.pitch for n in tail[-4:]] if tail else []
    ioi = [tail[i+1].start - tail[i].start for i in range(len(tail)-1)] if len(tail) > 1 else []
    rhythmic_variance = float(np.std(ioi)) if ioi else 0.0
    return {
        'note_count': len(notes),
        'avg_pitch': avg_pitch,
        'pitch_span': pitch_span,
        'density': density_bucket(notes_per_bar),
        'notes_per_bar': notes_per_bar,
        'register': register_bucket(avg_pitch),
        'bars': bars,
        'motif': motif,
        'rhythmic_variance': rhythmic_variance,
    }

features = extract_features(notes)
features


{'note_count': 126,
 'avg_pitch': 60.06349206349206,
 'pitch_span': 36,
 'density': 'medium',
 'notes_per_bar': 5.25,
 'register': 'mid',
 'bars': 24,
 'motif': [60, 65, 67, 79],
 'rhythmic_variance': 0.3273404954181583}

In [20]:
MEMORY = {
    "notes": [],
    "last_motif": None,
    "max_bars": 4
}

def quantize(t, step):
    return round(t / step) * step

def update_memory(memory, new_notes, tempo, time_sig):
    bar_len = seconds_per_bar(tempo, time_sig)
    max_time = memory["max_bars"] * bar_len

    memory["notes"].extend(new_notes)

    if memory["notes"]:
        latest_time = max(n.end for n in memory["notes"])
        memory["notes"] = [
            n for n in memory["notes"]
            if n.end >= latest_time - max_time
        ]

from collections import Counter

def detect_key(notes):
    if not notes:
        return 60
    pcs = [n.pitch % 12 for n in notes]
    root = Counter(pcs).most_common(1)[0][0]
    return 60 + root

def get_major_scale(root):
    return [(root + i) % 12 for i in [0, 2, 4, 5, 7, 9, 11]]

def constrain_to_scale(pitches, scale_pc):
    out = []
    for p in pitches:
        pc = p % 12
        if pc not in scale_pc:
            closest = min(scale_pc, key=lambda x: abs(x - pc))
            p = p - pc + closest
        out.append(p)
    return out

## Response Policy

This is a simple rule table for the MVP. It is not a learned model.


In [21]:
def decide_action(features):
    density = features['density']
    register = features['register']
    span = features['pitch_span']

    if density == 'high' and register == 'mid' and span >= 20:
        mode = 'contrast'
        response_density = 'low'
        octave_shift = 12
    elif density == 'high' and register == 'low':
        mode = 'sequence'
        response_density = 'medium'
        octave_shift = 24
    elif density == 'medium' and register == 'high':
        mode = 'fragment'
        response_density = 'medium'
        octave_shift = 12
    elif features['rhythmic_variance'] > 0.05:
        mode = 'repeat'
        response_density = 'medium'
        octave_shift = 0
    else:
        mode = 'sequence'
        response_density = 'low'
        octave_shift = 12

    return {
        'mode': mode,
        'response_density': response_density,
        'bars': features['bars'],
        'latency_bars': LATENCY_BARS,
        'target_notes': 32,
        'octave_shift': octave_shift,
    }

action = decide_action(features)
action


{'mode': 'repeat',
 'response_density': 'medium',
 'bars': 24,
 'latency_bars': 1,
 'target_notes': 32,
 'octave_shift': 0}

## Generate Response

The response starts after one bar of latency and uses a simple D minor palette for the first prototype.


In [22]:
def choose_response_program(source_instrument, action, features):
    # Keep the same instrument as the source for this MVP.
    return source_instrument.program


def build_response(notes, action, source_instrument, features, tempo=TEMPO, time_sig=TIME_SIG, start_time=0.0):
    response = pretty_midi.PrettyMIDI(initial_tempo=tempo)
    response_program = choose_response_program(source_instrument, action, features)
    inst = pretty_midi.Instrument(program=response_program)

    all_notes = MEMORY["notes"] + notes
    recent_notes = sorted(all_notes, key=lambda n: n.start)[-8:]
    motif = [n.pitch for n in recent_notes[-4:]] if recent_notes else []
    motif = motif if motif else [60, 62, 64, 67]

    key_root = detect_key(all_notes)
    scale_pc = get_major_scale(key_root)

    bar_len = seconds_per_bar(tempo, time_sig)
    step = (60.0 / tempo) / 2
    t = quantize(start_time + action['latency_bars'] * bar_len, step)

    if action['response_density'] == 'low':
        note_length = step * 1.5
    elif action['response_density'] == 'medium':
        note_length = step * 1.0
    else:
        note_length = step * 0.75

    base = motif[:]
    if action['mode'] == 'repeat':
        pitches = base + base[:4]
    elif action['mode'] == 'fragment':
        fragment = base[-2:]
        pitches = fragment * 8
    elif action['mode'] == 'sequence':
        pitches = base + [p + 2 for p in base] + [p + 4 for p in base]
    else:
        pitches = [p for p in base] + [p + 5 for p in base] + [p - 2 for p in base]

    pitches = [p + action['octave_shift'] for p in pitches]

    if notes:
        avg_pitch = int(np.mean([n.pitch for n in notes]))
        target_center = avg_pitch + (12 if action['mode'] != 'repeat' else 0)
        current_center = int(np.mean(pitches))
        center_shift = target_center - current_center
        pitches = [p + center_shift for p in pitches]

    pitches = constrain_to_scale(pitches, scale_pc)
    pitches = [int(np.clip(p, 48, 84)) for p in pitches]

    if action['response_density'] == 'low':
        pitches = pitches[:max(8, len(pitches) // 2)]

    target_notes = action.get('target_notes', 32)
    while len(pitches) < target_notes:
        pitches.extend([p + 2 for p in pitches[:4]])
    pitches = pitches[:target_notes]

    velocities = [70 + int(8 * np.sin(i / 2)) for i in range(len(pitches))]

    for idx, pitch in enumerate(pitches):
        note = pretty_midi.Note(
            velocity=int(np.clip(velocities[idx], 50, 100)),
            pitch=int(pitch),
            start=t,
            end=t + note_length,
        )
        inst.notes.append(note)
        t += step

    response.instruments.append(inst)
    MEMORY["last_motif"] = motif
    update_memory(MEMORY, notes, tempo, time_sig)
    return response


## Write Output

The output file can be imported into a DAW or opened in a MIDI player for evaluation.


In [23]:
response_midi = build_response(
    notes,
    action,
    source_instrument=instrument,
    features=features,
    start_time=midi.get_end_time()
)


In [24]:
combined = pretty_midi.PrettyMIDI(initial_tempo=TEMPO)
combined.instruments = midi.instruments + response_midi.instruments
OUTPUT_MIDI.parent.mkdir(parents=True, exist_ok=True)
combined.write(str(OUTPUT_MIDI))

print(f'Wrote {OUTPUT_MIDI}')
print('Detected features:', features)
print('Chosen action:', action)
print('Response instrument:', pretty_midi.program_to_instrument_name(response_midi.instruments[0].program))
print('Response note count:', len(response_midi.instruments[0].notes))
print('Response pitch range:', min(n.pitch for n in response_midi.instruments[0].notes), '-', max(n.pitch for n in response_midi.instruments[0].notes))


Wrote /Users/adrientalbot/Desktop/ai-jam-partner/data/output_midi/mvp_minimalist_response.mid
Detected features: {'note_count': 126, 'avg_pitch': 60.06349206349206, 'pitch_span': 36, 'density': 'medium', 'notes_per_bar': 5.25, 'register': 'mid', 'bars': 24, 'motif': [60, 65, 67, 79], 'rhythmic_variance': 0.3273404954181583}
Chosen action: {'mode': 'repeat', 'response_density': 'medium', 'bars': 24, 'latency_bars': 1, 'target_notes': 32, 'octave_shift': 0}
Response instrument: Acoustic Grand Piano
Response note count: 32
Response pitch range: 52 - 75
